## Imports

This example follows a common CmmDFT diffusion workflow: first compute adsorption states at several pressures, then project the equilibrium density onto a one-dimensional diffusion coordinate, and finally extract the free-energy profile along that path. The imported objects define the host, guest, EOS, solver, and post-processing tools used in this pipeline.

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt

from cmmdft.units_constants import *

from cmmdft.system import NanoporousHost, SphericalLJGuest, DualModelGuest, GuestMixture
from cmmdft.program import Program
from cmmdft.solver import Picard, Anderson, QuasiNewton
from cmmdft.calculator import Calculator
from cmmdft.plotter import Plotter
from cmmdft.eos import PCSAFTEOS, PengRobinsonMixEOS
from cmmdft.guest_dict import SAFT_pars_dict, chk_dict, pars_dict, LJ_pars_dict

In [ ]:
crit_parameters_dict = {'CH4' :  (16.031*amu, 190.56*kelvin, 45.99*atm, 0.01142),
                        'C2H4' : (30.047*amu, 282.34*kelvin, 50.41*atm, 0.0866),
                        'C2H6' : (28.031*amu, 305.32*kelvin, 48.718*atm, 0.0993),
                        'C3H6' : (42.047*amu, 364.9*kelvin, 45.79*atm, 0.144),
                        'C3H8' : (44.063*amu, 369.8*kelvin, 42.6*atm, 0.1530),
                        'CO2' :  (43.990*amu, 304.25*kelvin, 73.8*atm, 0.225)}

## System setup

The host is the porous framework (here, ZIF-8), and the guest is the diffusant of interest. The guest is modeled with a `DualModelGuest`, which combines a molecular interaction model with the hard-sphere repulsion needed by the PC-SAFT free-energy functional. The selected grid spacing and path resolution are deliberately tighter than in a simple loading calculation, because the diffusion profile requires a well-resolved density distribution and a stable free-energy reconstruction.

In [ ]:
guestname = 'C2H6_UA'
mass, sigma, epsilon = (30.011*amu, 4.41*angstrom, 0.46209*kcalmol)

guestchk = 'InputFiles/Guest/C2H6/struct_CA.chk'
guestpar = 'InputFiles/Guest/C2H6/pars_vdw_CA_trappe.txt'
guest = DualModelGuest(guestname, mass, sigma, epsilon, struct=guestchk, par=guestpar, ffname='TRAPPE-DUAL')

hostname = 'ZIF8'
hostchk = 'InputFiles/Host/ZIF8/ZIF8_opened_struct.pdb'
hostpar = 'InputFiles/Host/ZIF8/AMBER_system.xml'
host = NanoporousHost(hostname, hostchk, hostpar,
                      unit_energy='kjmol', unit_distance='angstrom', unit_sigma='nanometer', unit_mass='amu') #it is important to specify the units of the input file

ff_suffix = 'AMBER_TRAPPE'
funct_suffix = 'tWBII_MFA_WDAc'

grid_suffix = '0_25' # a lower resolution is chosen for the speed of this example

In [ ]:
# thermodynamic properties
# We simulate the adsorption state over a pressure range and subsequently compute the free-energy path.
# The resulting diffusion profile depends on both the thermodynamic state and the chosen pore geometry.
temperature = 300 * kelvin
pressures = np.linspace(0.1, 50, 5) * bar

In [ ]:
# program initialization
# The Program object bundles the host, guest, EOS, free-energy functional, and grid into one workflow.
# This is the same structure used for adsorption calculations, but with a tighter grid because diffusion paths
# need a much better spatial resolution to resolve bottlenecks and accessible channels.
program = Program(prefix=f'OutputFiles/DiffusionPathExample', hostname=hostname, guestname=guestname, ff_suffix=ff_suffix, funct_suffix=funct_suffix, grid_suffix=grid_suffix)

program.set_system(host, guest)
program.set_grid(spacing=0.25 * angstrom)
program.set_eos('MBWR')
program.init_free_energy(temperature)

program.fener.add_external_potential(temperature)
program.fener.add_hard_sphere()
program.fener.add_mean_field()
program.fener.add_correlation_wda_lj()
program.set_temperature(temperature)

# Defining multiple convergence criteria for the self-consistent field iteration.
# The Anderson mixer accelerates convergence and is often more robust than a simple Picard loop.
criteria = ['riue', 'res']
thresholds = [1e-4, 1e-6]

solver = Anderson(program, criterion=criteria, threshold=thresholds)
program.set_solver(solver)

calc = Calculator(program)

In [ ]:
# Solve the equilibrium density profile for each pressure.
# The saved loading and grand potential are later used to reconstruct the free-energy landscape along the diffusion coordinate.
for P in pressures:
    program.solve(pressure=P)
    calc.save_loading_and_grand_potential(temperature=temperature, pressure=P) # Saves loading and grand potential per calculation to an npz file for later use (necessary for later integration)


## Defining the diffusion axis

The diffusion path is defined by a collective variable (CV) that tracks the molecule's position along a relevant channel or gate. In this example, the path follows the diagonal through the pore opening and is constructed from a set of atoms that define the narrowest region of the framework.

In [ ]:
# Definition of the diffusion coordinate.
# The selected carbon atoms define the gate opening of the framework; the path is then constructed along the 111 diagonal.
carbon_indices = np.array([171, 191, 175, 187, 179, 183])  # atoms defining the gate opening
step_dist = program.grid.spacings[0] / np.sqrt(3)  # diffusion axis is the 111 diagonal for this unit cell
distance_from_axis = 7 * angstrom  # keep the integration to the relevant pore region
cvs_limits = np.array([-7.5, 7.5]) * angstrom  # edges of the CV range

# Collect the one-dimensional coordinate and the geometric mask for the chosen diffusion path.
cvs, cvs_mat, dist_mask = calc.collective_variable(
    ring_indices=carbon_indices,
    dist_from_axis=distance_from_axis,
    step_dist=step_dist,
    cvs_limits=cvs_limits,
    supercell=True,
)

In [ ]:
# Project the equilibrium density onto the diffusion coordinate for each pressure.
# The free-energy path is then computed from the projected density, giving an effective barrier along the trajectory.
for P in pressures:
    calc.project_density(cvs, cvs_mat, dist_mask, temperature=temperature, pressure=P, rewrite=True)
    calc.free_energy_path(temperature=temperature, pressure=P)

In [ ]:
plotter = Plotter(calc)
fig = plotter.plot_free_energy_profile(temperature, pressures=pressures, relative=True)

In [ ]:
from cmmdft.tools import get_chempot_key

# Barrier height as a function of pressure: the barrier is the difference between the highest
# and lowest free-energy values along the diffusion coordinate.
barrier_by_pressure = []
for P in pressures:
    mu = calc.get_chempot_from_pressures(temperature, pressure=P)
    mu_key = get_chempot_key(mu)
    h5_path = f"T_{temperature / kelvin:0.5f}/mu_{mu_key}"

    with h5py.File(calc.workdir / 'projected_density_energy.hdf5', 'r') as h5:
        free = h5[h5_path]['free_energy'][:]

    if free.ndim == 2:
        free_profile = free[:, 0] if free.shape[1] > 1 else free.ravel()
    else:
        free_profile = np.asarray(free).ravel()

    barrier_by_pressure.append(np.max(free_profile) - np.min(free_profile))

fig_barrier, ax_barrier = plt.subplots(figsize=(7, 5))
ax_barrier.plot(pressures / bar, np.asarray(barrier_by_pressure) / kjmol, marker='o', color='tab:purple')
ax_barrier.set_xlabel('Pressure [bar]')
ax_barrier.set_ylabel('Barrier height [kJ/mol]')
ax_barrier.set_title('Diffusion barrier evolution with pressure')
ax_barrier.grid(True, alpha=0.3)
fig_barrier.tight_layout()

# Guest mixture

This is useful when studying competitive transport or selectivity in mixed adsorbates, where the barrier along a channel depends not only on the framework but also on the mixture composition.

The same diffusion-analysis workflow can be repeated for a guest mixture. In that case, the adsorbate is represented by multiple guest species with a fixed composition, and the diffusion coordinate is computed with respect to the same host geometry.

In [ ]:
guestname1 = 'C2H6_UA'
mass1, sigma1, epsilon1 = (30.011*amu, 4.41*angstrom, 0.46209*kcalmol)
guestchk1 = 'InputFiles/Guest/C2H6/struct_CA.chk'
guestpar1 = 'InputFiles/Guest/C2H6/pars_vdw_CA_trappe.txt'
guest1 = DualModelGuest(guestname1, mass1, sigma1, epsilon1, struct=guestchk1, par=guestpar1, ffname='TRAPPE-DUAL')

guestname2 = 'CH4_UA'
mass2, sigma2, epsilon2 = (16*amu, 3.75*angstrom, 0.29411*kcalmol)
guest2 = SphericalLJGuest(guestname2, mass2, sigma2, epsilon2, ffname='TRAPPE-DUAL')

mole_fractions = [0.5, 0.5]
mixture = GuestMixture([guest1, guest2], mole_fractions)

hostname = 'ZIF8'
hostchk = 'InputFiles/Host/ZIF8/ZIF8_opened_struct.pdb'
hostpar = 'InputFiles/Host/ZIF8/AMBER_system.xml'
host = NanoporousHost(hostname, hostchk, hostpar,
                      unit_energy='kjmol', unit_distance='angstrom', unit_sigma='nanometer', unit_mass='amu') #it is important to specify the units of the input file


ff_suffix = 'AMBER_TRAPPE'
funct_suffix = 'atWBII_MFA_WDAc'

grid_suffix = '0_5'

In [ ]:
# thermodynamic properties
# We simulate the adsorption state over a pressure range and subsequently compute the free-energy path.
# The resulting diffusion profile depends on both the thermodynamic state and the chosen pore geometry.
temperature = 300 * kelvin
pressures = np.linspace(0.1, 50, 5) * bar

In [ ]:
# program initialization
program = Program(prefix=f'OutputFiles/DiffusionPathExample', hostname=hostname, guestname=mixture.name, ff_suffix=ff_suffix, funct_suffix=funct_suffix, grid_suffix=grid_suffix)

program.set_system(host, mixture)
program.set_grid(spacing = 0.5*angstrom)
program.set_eos('MBWR')
program.init_free_energy(temperature)

program.fener.add_external_potential(temperature)
program.fener.add_hard_sphere(version='atWBII')
program.fener.add_mean_field()
program.fener.add_correlation_wda_lj()
program.set_temperature(temperature)

#defining multiple convergence criteria
criteria = ['riue', 'res']
thresholds = [1e-4, 1e-6]

solver = Anderson(program, criterion=criteria, threshold=thresholds)
program.set_solver(solver)

calc = Calculator(program)

In [ ]:
# Solve the equilibrium density profile for each pressure.
# The saved loading and grand potential are later used to reconstruct the free-energy landscape along the diffusion coordinate.
for P in pressures:
    program.solve(pressure=P)
    calc.save_loading_and_grand_potential(temperature=temperature, pressure=P) # Saves loading and grand potential per calculation to an npz file for later use (necessary for later integration)


In [ ]:
# Definition of the diffusion coordinate.
# The selected carbon atoms define the gate opening of the framework; the path is then constructed along the 111 diagonal.
carbon_indices = np.array([171, 191, 175, 187, 179, 183])  # atoms defining the gate opening
step_dist = program.grid.spacings[0] / np.sqrt(3)  # diffusion axis is the 111 diagonal for this unit cell
distance_from_axis = 7 * angstrom  # keep the integration to the relevant pore region
cvs_limits = np.array([-7.5, 7.5]) * angstrom  # edges of the CV range

# Collect the one-dimensional coordinate and the geometric mask for the chosen diffusion path.
cvs, cvs_mat, dist_mask = calc.collective_variable(
    ring_indices=carbon_indices,
    dist_from_axis=distance_from_axis,
    step_dist=step_dist,
    cvs_limits=cvs_limits,
    supercell=True,
)

In [ ]:
# Project the equilibrium density onto the diffusion coordinate for each pressure.
# The free-energy path is then computed from the projected density, giving an effective barrier along the trajectory.
for P in pressures:
    calc.project_density(cvs, cvs_mat, dist_mask, temperature=temperature, pressure=P)
    calc.free_energy_path(temperature=temperature, pressure=P)

In [ ]:
plotter = Plotter(calc)
fig = plotter.plot_free_energy_profile(temperature, pressures=pressures[::3], relative=True)


In [ ]:

# Barrier height as a function of pressure: the barrier is the difference between the highest
# and lowest free-energy values along the diffusion coordinate.
barrier_by_pressure = []
for P in pressures:
    mu = calc.get_chempot_from_pressures(temperature, pressure=P)
    mu_key = f"{mu / kjmol:0.5f}"
    h5_path = f"T_{temperature / kelvin:0.5f}/mu_{mu_key}"

    with h5py.File(calc.workdir / 'projected_density_energy.hdf5', 'r') as h5:
        free = h5[h5_path]['free_energy'][:]

    barrier_by_pressure.append(np.max(free_profile, axis=0) - np.min(free_profile, axis=0))

fig_barrier, ax_barrier = plt.subplots(figsize=(7, 5))
ax_barrier.plot(pressures / bar, np.asarray(barrier_by_pressure) / kjmol, marker='o', color='tab:purple')
ax_barrier.set_xlabel('Pressure [bar]')
ax_barrier.set_ylabel('Barrier height [kJ/mol]')
ax_barrier.set_title('Diffusion barrier evolution with pressure')
ax_barrier.grid(True, alpha=0.3)
fig_barrier.tight_layout()